# Document Pre-processing for Knowledge Tuning

## Overview

This notebook demonstrates a complete document preprocessing pipeline designed specifically for **knowledge tuning** with sdg-hub. 

## What This Notebook Does

This preprocessing pipeline transforms raw documents (PDFs, Word docs, etc.) into seed data for data generation:

1. **Document Parsing**: Converts raw documents to structured markdown format
2. **Chunking**: Splits documents into manageable chunks while preserving structure and context
3. **Seed Data Creation**: Formats chunks with in-context learning (ICL) templates for effective knowledge tuning

## Prerequisites

- We will use the existing InstructLab document parser (`docparser_v2.py`) and Document parsing configuration (`docling_v2_config.yaml`)
- Raw pdf documents in the `document_collection/` directory


In [ ]:
# Step 1: Document Processing Pipeline
# Define the directory containing raw documents to be processed
data_dir = "document_collection/"

# Run the document parser to convert documents to markdown
# - input-dir: Directory containing source documents
# - output-dir: Directory where processed markdown files will be saved
# - c: Configuration file specifying parsing parameters
!python ../docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c ../docling_v2_config.yaml

In [ ]:
# Step 2: Install Required Dependencies
# Install packages needed for document processing and text chunking

#%pip install docling markdown-it-py
#%pip install --upgrade transformers
#%pip install litellm python-dotenv

In [ ]:
# Step 3 & 4: Load Documents, Chunk, and Generate Dynamic ICL
import os
import glob
import json
from typing import List
import datasets
from litellm import completion
from markdown_it import MarkdownIt
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Multilingual configuration
sdg_lang = os.getenv("SDG_LANG", "Portuguese (Brazil)")
sdg_lang_code = os.getenv("SDG_LANG_CODE", "pt-br")

# Model configuration for ICL generation (Translator Model or VLLM fallback)
model_name = os.getenv("TRANSLATOR_MODEL", os.getenv("VLLM_MODEL", "hosted_vllm/openai/RedHatAI/gpt-oss-20b"))
api_base = os.getenv("TRANSLATOR_API_BASE", os.getenv("VLLM_API_BASE"))
api_key = os.getenv("TRANSLATOR_API_KEY", os.getenv("VLLM_API_KEY", "EMPTY"))

def chunk_markdown(text: str, max_tokens: int = 200, overlap: int = 50) -> List[str]:
    """Splits Markdown text into chunks at block-level elements."""
    md = MarkdownIt()
    tokens = md.parse(text)
    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
                buf = []
        elif tok.content:
            buf.append(tok.content)
    if buf:
        blocks.append("\n".join(buf).strip())

    chunks = []
    current_words = []
    for block in blocks:
        words = block.split()
        for w in words:
            current_words.append(w)
            if len(current_words) >= max_tokens:
                chunks.append(" ".join(current_words))
                current_words = current_words[-overlap:] if overlap > 0 else []
    if current_words:
        chunks.append(" ".join(current_words))
    return chunks

all_records = []
md_files = glob.glob(f"{data_dir}/*.md")
print(f"Found {len(md_files)} processed markdown files in {data_dir}.")

for file_path in md_files:
    print(f"\nProcessing file: {file_path}")
    with open(file_path, "r", encoding="utf-8") as f:
        text = f.read()
        
    chunks = chunk_markdown(text, max_tokens=5000, overlap=1000)
    if not chunks:
        continue
        
    # Get a sample paragraph (long enough to make sense) for the LLM to generate the ICL questions.
    sample_text = next((chunk for chunk in chunks if len(chunk) > 100), chunks[0])
    
    prompt = f"""
Você é um especialista em treinamento de LLMs. Analise o seguinte documento e crie exatamente 3 perguntas 
valiosas e desafiadoras que possam ser feitas com base neste texto.
O idioma das perguntas e resumo deve ser estritamente: {sdg_lang} ({sdg_lang_code}).
Responda APENAS com um objeto JSON válido contendo as seguintes chaves:
"document_outline", "icl_query_1", "icl_query_2", "icl_query_3", "domain"

Texto do Documento:
{sample_text[:2000]}

Formato JSON esperado:
{{
  "document_outline": "Resumo de 1 frase descrevendo o escopo deste documento.",
  "icl_query_1": "Sua pergunta numero 1?",
  "icl_query_2": "Sua pergunta numero 2?",
  "icl_query_3": "Sua pergunta numero 3?",
  "domain": "Domínio do texto (ex: Finanças, Leis, TI, etc)"
}}
"""
    print(f"  -> Generating dynamic ICL using model: {model_name}...")
    try:
        response = completion(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            api_base=api_base,
            api_key=api_key,
            temperature=0.3,
            response_format={"type": "json_object"}
        )
        
        llm_output = response.choices[0].message.content
        generated_data = json.loads(llm_output)
        
        icl = {
            "document_outline": generated_data.get("document_outline", "Resumo do Documento"),
            "icl_document": sample_text,
            "icl_query_1": generated_data.get("icl_query_1", "Pergunta 1?"),
            "icl_query_2": generated_data.get("icl_query_2", "Pergunta 2?"),
            "icl_query_3": generated_data.get("icl_query_3", "Pergunta 3?"),
            "domain": generated_data.get("domain", "Geral")
        }
        print("  -> ICL Generated successfully!")
        
    except Exception as e:
        print(f"  -> Error generating ICL, using fallback template. Error: {e}")
        icl = {
            "document_outline": "Resumo do Documento (Fallback)",
            "icl_document": sample_text,
            "icl_query_1": "Qual é o tema principal deste texto?",
            "icl_query_2": "Quais são as principais regras ou pontos abordados?",
            "icl_query_3": "Para quem este documento é destinado?",
            "domain": "Geral"
        }
        
    # Append all chunks for this file using the generated file-specific ICL
    for chunk in chunks:
        record = {"document": chunk}
        record.update(icl)
        all_records.append(record)

# Create single unified HF Dataset
seed_data = datasets.Dataset.from_list(all_records)

# Save to final output file
seed_data.to_json("seed_data.jsonl", orient="records", lines=True)
print(f"\nSuccessfully compiled {len(seed_data)} total chunks into seed_data.jsonl!")


### Next Steps:
- The seed_data.jsonl file is now ready for the knowledge tuning pipeline.
- You can now refer to the [knowledge generation](knowledge_generation.ipynb) notebook